# Module 01 — Mini Project
# Personal Expenses Analysis

---

**Your name:**  *(write here)*  
**Date range analysed:**  *(e.g., 8 Jan 2024 – 14 Jan 2024)*  
**Dataset used:**  `sample_expenses.csv` *(change to `my_expenses.csv` when using your own data)*

---

Most people have no idea where their money actually goes. They feel like they're spending carefully — and then wonder why their account is empty by the 20th of the month.

Data science can fix that. In this project, you will apply the **full 7-step data science lifecycle** to a real dataset: your own weekly expenses. You'll move from raw numbers to clear insights using only Python built-ins — no libraries yet.

By the end, you'll have a written report you can actually act on.

---
## Setup — Configure Your Dataset

Change the filename below to switch between the sample data and your own data.

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Change this to "data/my_expenses.csv" once you have your own data ready
FILENAME       = "data/sample_expenses.csv"
WEEKLY_BUDGET  = 3000   # ₹ — your target spending for the week (adjust to your situation)
CURRENCY       = "₹"
# ─────────────────────────────────────────────────────────────────────────────

---
## Step 1 — Define the Problem

**Before looking at a single row of data, write down what you want to know.**

This is the most important habit in data science. Once you open the data, your questions will get pulled in whatever direction the data takes you. Lock in your questions first.

### Our five questions for this project:

| # | Question | Type of answer |
|---|---|---|
| Q1 | How much did I spend in total this week? | One number |
| Q2 | Which category takes the largest share of my budget? | Category + percentage |
| Q3 | Which day did I spend the most? | Day + amount |
| Q4 | How does my spending vary day by day? | A trend / pattern |
| Q5 | Are there any unusual one-off spikes? | A list of outlier transactions |

### Success criteria
This project is successful if I can give a specific, number-backed answer to each of the five questions — and walk away with at least one thing I'll do differently next week.

---
## Step 2 — Collect Data

In a real project, this is where you connect to a database or API. Here, the data is a CSV file you either downloaded or filled in yourself. Let's load it.

In [ ]:
import csv
import statistics
from collections import defaultdict

def load_expenses(filename):
    """Reads the CSV and returns a list of expense dictionaries."""
    expenses = []
    with open(filename, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            # Convert amount from string to float as we load
            row["amount"] = float(row["amount"])
            expenses.append(row)
    return expenses

expenses = load_expenses(FILENAME)

print(f"Loaded {len(expenses)} expense records from '{FILENAME}'")
print()
print(f"{'Date':<13} {'Category':<15} {'Description':<35} {'Amount':>8}")
print("─" * 75)
for row in expenses[:8]:   # preview first 8 rows
    print(f"{row['date']:<13} {row['category']:<15} {row['description']:<35} {CURRENCY}{row['amount']:>7.0f}")
print(f"... ({len(expenses) - 8} more rows)")

---
## Step 3 — Clean the Data

Before analysing anything, check the data for problems: missing values, duplicate rows, wrong types, inconsistent category names.

In [ ]:
# ── Check 1: Are there any missing values? ────────────────────────────────────
columns = ["date", "category", "description", "amount"]
print("Missing value check:")
issues_found = False
for col in columns:
    missing = sum(1 for row in expenses if not row.get(col) and row.get(col) != 0)
    status  = f"⚠  {missing} missing" if missing > 0 else "✓  none missing"
    print(f"  {col:<15} {status}")
    if missing > 0:
        issues_found = True

if not issues_found:
    print("\nAll columns complete. No missing values to handle.")

In [ ]:
# ── Check 2: Negative or zero amounts ─────────────────────────────────────────
# Zero is valid only for expenses like "Lunch at home (₹0)"
# Negative amounts would indicate a refund — flag them for review

zero_rows     = [r for r in expenses if r["amount"] == 0]
negative_rows = [r for r in expenses if r["amount"] < 0]

print(f"Zero-amount rows    : {len(zero_rows)}")
for r in zero_rows:
    print(f"  {r['date']}  {r['description']:<30}  {CURRENCY}{r['amount']}")

print(f"\nNegative-amount rows: {len(negative_rows)}")
for r in negative_rows:
    print(f"  {r['date']}  {r['description']:<30}  {CURRENCY}{r['amount']}")

if not negative_rows:
    print("  None — good.")

In [ ]:
# ── Check 3: Consistent category names ────────────────────────────────────────
# Typos like "food" vs "Food" or "transport " (trailing space) would split a category in two

from collections import Counter

category_counts = Counter(row["category"] for row in expenses)
print("Categories found (check for typos or duplicates):")
for cat, count in sorted(category_counts.items()):
    print(f"  '{cat}'  →  {count} transactions")

# Check for suspicious near-duplicates (lowercase comparison)
lower_cats = [cat.lower().strip() for cat in category_counts]
if len(set(lower_cats)) < len(category_counts):
    print("\n⚠  Possible duplicate categories detected (same name, different case/spacing)")
else:
    print("\n✓  No duplicate categories detected.")

In [ ]:
# ── Check 4: Date range ────────────────────────────────────────────────────────
dates = sorted(set(row["date"] for row in expenses))
print(f"Date range  : {dates[0]}  →  {dates[-1]}")
print(f"Days covered: {len(dates)}")
print(f"Dates       : {dates}")

# Check for any expected dates missing in the range
# (simple check — assumes dates are YYYY-MM-DD and consecutive)
from datetime import date, timedelta

start = date.fromisoformat(dates[0])
end   = date.fromisoformat(dates[-1])
expected = [(start + timedelta(days=i)).isoformat() for i in range((end - start).days + 1)]
missing_dates = [d for d in expected if d not in dates]

if missing_dates:
    print(f"\n⚠  Missing dates (no expenses recorded): {missing_dates}")
    print("   This is fine if you genuinely spent ₹0 those days.")
else:
    print("\n✓  All days in the range have at least one entry.")

### Cleaning Summary

*(Fill this in after running the checks above)*

| Check | Result | Action taken |
|---|---|---|
| Missing values | | |
| Negative amounts | | |
| Category consistency | | |
| Date range | | |

In [ ]:
# ── Apply any cleaning decisions here ─────────────────────────────────────────
# Example: remove zero-amount rows from analysis (they represent free items)
# Uncomment the line below if you want to exclude ₹0 entries

# expenses = [r for r in expenses if r["amount"] > 0]

# Standardise category names (strip whitespace, title case)
for row in expenses:
    row["category"] = row["category"].strip().title()

print(f"Clean dataset ready: {len(expenses)} rows.")

---
## Step 4 — Explore the Data

Before answering specific questions, get a broad feel for the data. What's the range of amounts? How many entries per day? Any patterns jumping out?

In [ ]:
# ── Overview statistics ────────────────────────────────────────────────────────
amounts = [r["amount"] for r in expenses if r["amount"] > 0]

print("=" * 50)
print("  DATASET OVERVIEW")
print("=" * 50)
print(f"  Total transactions : {len(expenses)}")
print(f"  Days covered       : {len(dates)}")
print(f"  Categories         : {len(category_counts)}")
print()
print("  Per-transaction amount:")
print(f"    Smallest  : {CURRENCY}{min(amounts):.0f}")
print(f"    Largest   : {CURRENCY}{max(amounts):.0f}")
print(f"    Average   : {CURRENCY}{statistics.mean(amounts):.0f}")
print(f"    Median    : {CURRENCY}{statistics.median(amounts):.0f}")
print(f"    Std dev   : {CURRENCY}{statistics.stdev(amounts):.0f}")
print()
print("  Observations per day:")
daily_counts = Counter(r["date"] for r in expenses)
for d in sorted(daily_counts):
    bar = "■" * daily_counts[d]
    print(f"    {d}  {bar}  ({daily_counts[d]} transactions)")

In [ ]:
# ── Distribution of transaction sizes ─────────────────────────────────────────
# How many transactions are tiny (< ₹50), medium (₹50–200), large (> ₹200)?

bins = [
    ("Under ₹50",    0,   50),
    ("₹50 – ₹100",  50,  100),
    ("₹100 – ₹200", 100, 200),
    ("₹200 – ₹500", 200, 500),
    ("Over ₹500",   500, float("inf")),
]

print("Transaction size distribution:")
print()
total_txns = len(amounts)
for label, low, high in bins:
    count    = sum(1 for a in amounts if low <= a < high)
    pct      = count / total_txns * 100
    bar      = "█" * count
    print(f"  {label:<15}  {bar:<20}  {count:>2} transactions  ({pct:.0f}%)")

---
## Step 5 — Analyse

Now we answer each of the five questions we defined in Step 1.

### Q1 — How much did I spend in total this week?

In [ ]:
total_spent = sum(r["amount"] for r in expenses)
budget_left = WEEKLY_BUDGET - total_spent
budget_pct  = total_spent / WEEKLY_BUDGET * 100

print(f"Total spent this week : {CURRENCY}{total_spent:,.0f}")
print(f"Weekly budget         : {CURRENCY}{WEEKLY_BUDGET:,.0f}")
print()

if budget_left >= 0:
    print(f"Under budget by       : {CURRENCY}{budget_left:,.0f}  ({100 - budget_pct:.0f}% remaining)  ✓")
else:
    print(f"Over budget by        : {CURRENCY}{abs(budget_left):,.0f}  ({budget_pct - 100:.0f}% over limit)  ⚠")

# Visual budget bar
print()
filled = min(int(budget_pct / 5), 20)
empty  = 20 - filled
bar    = "█" * filled + "░" * empty
print(f"  Budget used: [{bar}] {budget_pct:.0f}%")

### Q2 — Which category takes the largest share?

In [ ]:
# Aggregate total and count by category
category_totals = defaultdict(float)
category_txns   = defaultdict(int)

for row in expenses:
    category_totals[row["category"]] += row["amount"]
    category_txns[row["category"]]   += 1

# Sort by total amount, highest first
sorted_cats = sorted(category_totals.items(), key=lambda x: -x[1])

print(f"{'Category':<16} {'Spent':>8}  {'Share':>6}  {'Txns':>5}  {'Avg/txn':>8}")
print("─" * 55)

for cat, total in sorted_cats:
    pct     = total / total_spent * 100
    n_txns  = category_txns[cat]
    avg_txn = total / n_txns
    print(f"{cat:<16} {CURRENCY}{total:>7.0f}  {pct:>5.1f}%  {n_txns:>5}  {CURRENCY}{avg_txn:>7.0f}")

print("─" * 55)
print(f"{'TOTAL':<16} {CURRENCY}{total_spent:>7.0f}  100.0%")

top_cat       = sorted_cats[0][0]
top_cat_total = sorted_cats[0][1]
top_cat_pct   = top_cat_total / total_spent * 100

print()
print(f"→ Biggest category: '{top_cat}' at {CURRENCY}{top_cat_total:.0f} ({top_cat_pct:.1f}% of total)")

In [ ]:
# Visual: spending by category as a horizontal bar chart
print("\nCategory breakdown:")
print()
max_total = sorted_cats[0][1]
for cat, total in sorted_cats:
    pct      = total / total_spent * 100
    bar_len  = int((total / max_total) * 30)
    bar      = "█" * bar_len
    print(f"  {cat:<15} {bar:<30}  {CURRENCY}{total:>6.0f}  ({pct:.0f}%)")

### Q3 — Which day did I spend the most?

In [ ]:
# Aggregate spending by date
daily_totals = defaultdict(float)
for row in expenses:
    daily_totals[row["date"]] += row["amount"]

sorted_days = sorted(daily_totals.items())  # chronological order
day_amounts = [total for _, total in sorted_days]

max_day     = max(sorted_days, key=lambda x: x[1])
min_day     = min(sorted_days, key=lambda x: x[1])
daily_avg   = statistics.mean(day_amounts)

print(f"{'Date':<13} {'Spent':>8}  {'vs Daily Avg':>12}")
print("─" * 40)

for d, total in sorted_days:
    diff   = total - daily_avg
    flag   = "  ← highest" if d == max_day[0] else ("  ← lowest" if d == min_day[0] else "")
    sign   = "+" if diff >= 0 else "-"
    print(f"{d:<13} {CURRENCY}{total:>7.0f}  ({sign}{CURRENCY}{abs(diff):>6.0f}){flag}")

print("─" * 40)
print(f"{'Daily average':<13} {CURRENCY}{daily_avg:>7.0f}")
print()
print(f"→ Highest spending day : {max_day[0]}  ({CURRENCY}{max_day[1]:.0f})")
print(f"→ Lowest spending day  : {min_day[0]}  ({CURRENCY}{min_day[1]:.0f})")

### Q4 — How does my spending vary day by day?

In [ ]:
# Day-by-day stacked breakdown: show each category's contribution per day
# This shows WHERE money is going on each day, not just the total

# Get unique categories sorted by total (most expensive first)
cat_order = [cat for cat, _ in sorted_cats]

# Build day × category grid
daily_by_cat = defaultdict(lambda: defaultdict(float))
for row in expenses:
    daily_by_cat[row["date"]][row["category"]] += row["amount"]

print("Daily spending by category:")
print()

# Header
cat_abbr = {cat: cat[:6] for cat in cat_order}
header = f"  {'Date':<13}" + "".join(f"{cat_abbr[c]:>9}" for c in cat_order) + f"  {'TOTAL':>8}"
print(header)
print("  " + "─" * (13 + 9 * len(cat_order) + 10))

for d, total in sorted_days:
    row_str = f"  {d:<13}"
    for cat in cat_order:
        val = daily_by_cat[d].get(cat, 0)
        row_str += f"  {CURRENCY}{val:>6.0f}" if val > 0 else f"  {'—':>7}"
    row_str += f"  {CURRENCY}{total:>6.0f}"
    print(row_str)

In [ ]:
# Simpler view: ASCII daily spend bar chart
print("\nDaily spending (bar chart):")
print(f"  Each █ ≈ {CURRENCY}25\n")

scale = 25
for d, total in sorted_days:
    bar      = "█" * int(total / scale)
    over_avg = " ←" if total > daily_avg * 1.3 else ""
    print(f"  {d}  {bar:<35}  {CURRENCY}{total:>6.0f}{over_avg}")

# Draw average line indicator
avg_bar_len = int(daily_avg / scale)
print()
print(f"  Daily average ({CURRENCY}{daily_avg:.0f}):  " + " " * avg_bar_len + "▲")

### Q5 — Are there any unusual one-off spikes?

In [ ]:
# Outlier detection using the IQR method
# Any transaction more than 1.5× IQR above Q3 is flagged as unusual
# You'll learn this properly in Module 6 (Data Cleaning)

sorted_amounts = sorted(amounts)
n = len(sorted_amounts)

q1_idx = n // 4
q3_idx = 3 * n // 4
q1     = sorted_amounts[q1_idx]
q3     = sorted_amounts[q3_idx]
iqr    = q3 - q1
upper_fence = q3 + 1.5 * iqr

print(f"Outlier detection (IQR method):")
print(f"  Q1 (25th percentile)   : {CURRENCY}{q1:.0f}")
print(f"  Q3 (75th percentile)   : {CURRENCY}{q3:.0f}")
print(f"  IQR (Q3 - Q1)          : {CURRENCY}{iqr:.0f}")
print(f"  Upper fence (Q3+1.5×IQR): {CURRENCY}{upper_fence:.0f}")
print()

outliers = [r for r in expenses if r["amount"] > upper_fence]

if outliers:
    print(f"Flagged transactions (unusually large for your spending pattern):")
    print(f"  {'Date':<13} {'Category':<15} {'Description':<30} {'Amount':>8}")
    print("  " + "─" * 72)
    for r in sorted(outliers, key=lambda x: -x["amount"]):
        print(f"  {r['date']:<13} {r['category']:<15} {r['description']:<30} {CURRENCY}{r['amount']:>7.0f}")
    print()
    print("  These aren't necessarily bad — but they're worth noticing.")
    print("  Ask: Was this planned? Could it be split or reduced?")
else:
    print("No statistical outliers detected — your spending is relatively consistent.")

---
## Step 6 — Evaluate

Before presenting anything, sanity-check the results. Do the numbers add up? Do the insights match reality?

In [ ]:
# ── Sanity checks ─────────────────────────────────────────────────────────────
print("Sanity checks:")
print()

# Check 1: Category totals sum to overall total
cat_sum = sum(category_totals.values())
match   = abs(cat_sum - total_spent) < 0.01   # floating point tolerance
print(f"  Sum of category totals = {CURRENCY}{cat_sum:.2f}")
print(f"  Overall total          = {CURRENCY}{total_spent:.2f}")
print(f"  Match: {'✓ Yes' if match else '✗ No — check your data!'}")
print()

# Check 2: Daily totals sum to overall total
day_sum = sum(daily_totals.values())
match2  = abs(day_sum - total_spent) < 0.01
print(f"  Sum of daily totals    = {CURRENCY}{day_sum:.2f}")
print(f"  Match: {'✓ Yes' if match2 else '✗ No — check your data!'}")
print()

# Check 3: Category percentages sum to 100%
pct_sum = sum(v / total_spent * 100 for v in category_totals.values())
print(f"  Category percentages sum: {pct_sum:.1f}%  {'✓' if abs(pct_sum - 100) < 0.1 else '✗'}")
print()

# Check 4: Does the highest-spend day match intuition?
print(f"  Highest spend day: {max_day[0]} ({CURRENCY}{max_day[1]:.0f})")
print(f"  → Look at the transactions for this day — does it make sense?")
print()
high_day_txns = [r for r in expenses if r["date"] == max_day[0]]
for r in high_day_txns:
    print(f"      {r['category']:<15} {r['description']:<30} {CURRENCY}{r['amount']:.0f}")

---
## Step 7 — Communicate

This is the most important section. Numbers without interpretation are useless. Write your findings clearly enough that someone who hasn't seen the data can read this and understand what to do.

**Fill in the section below with your actual findings after running the notebook.**

In [ ]:
# ── Auto-generated report skeleton ────────────────────────────────────────────
# This prints the key numbers — add your written interpretation below each section

print("=" * 60)
print("   WEEKLY EXPENSE REPORT")
print(f"   Period: {dates[0]}  to  {dates[-1]}")
print("=" * 60)

print()
print("1. OVERVIEW")
print(f"   Total spent  : {CURRENCY}{total_spent:,.0f} / {CURRENCY}{WEEKLY_BUDGET:,.0f} budget")
status_word = "under" if budget_left >= 0 else "OVER"
print(f"   Status       : {status_word} budget by {CURRENCY}{abs(budget_left):.0f}")
print(f"   Transactions : {len(expenses)} across {len(dates)} days")

print()
print("2. WHERE THE MONEY WENT")
for cat, total in sorted_cats[:3]:   # top 3 categories
    pct = total / total_spent * 100
    print(f"   {cat:<15} {CURRENCY}{total:>6.0f}  ({pct:.0f}%)")
if len(sorted_cats) > 3:
    rest_total = sum(v for _, v in sorted_cats[3:])
    rest_pct   = rest_total / total_spent * 100
    print(f"   {'Others':<15} {CURRENCY}{rest_total:>6.0f}  ({rest_pct:.0f}%)")

print()
print("3. DAY-BY-DAY")
for d, total in sorted_days:
    bar  = "█" * int(total / 50)
    flag = " ← peak" if d == max_day[0] else ""
    print(f"   {d}  {bar:<20}  {CURRENCY}{total:>6.0f}{flag}")

print()
print("4. UNUSUAL TRANSACTIONS")
if outliers:
    for r in sorted(outliers, key=lambda x: -x["amount"]):
        print(f"   {r['date']}  {r['description']:<30}  {CURRENCY}{r['amount']:.0f}")
else:
    print("   No statistical outliers this week.")

print()
print("=" * 60)

### Your Written Insights

*(Replace the placeholder text below with your real analysis. This is what you would actually present to someone.)*

---

**Q1 — Total spending:**  
This week I spent ₹_____ out of my ₹_____ budget, which means I was [under/over] budget by ₹_____. [Add one sentence about whether this is typical for you or unusual.]

---

**Q2 — Biggest category:**  
[Category name] was my biggest expense at ₹_____ (____%). [Is this a fixed or variable cost? Could it be reduced? Is it essential?]

---

**Q3 — Biggest day:**  
[Date / day name] was my most expensive day at ₹_____. This was because [explain what drove the spending — a specific event, an impulse purchase, a planned expense?].

---

**Q4 — Day-by-day pattern:**  
[Describe the pattern. Are weekdays consistent and weekends higher? Are there specific days that are always higher?]

---

**Q5 — Unusual transactions:**  
[Were there any outlier transactions? Were they planned or impulse? What would have happened to your budget without them?]

---

**One thing I'll do differently next week:**  
[Specific, actionable — not "spend less" but "I'll set a ₹500 limit for weekend outings" or "I'll meal-prep on Sunday to reduce food spending Monday-Wednesday."]

---

**The most surprising thing I found:**  
[What surprised you? This is often the most valuable insight — the thing you didn't expect to see in your own data.]

---
## Bonus — Extended Analysis

Finished the main project? Try these extensions:

In [ ]:
# BONUS 1 — Per-category daily average
# How much do I spend on Food per day on average?

print("Average daily spend per category:")
n_days = len(dates)
for cat, total in sorted_cats:
    daily_avg_cat = total / n_days
    print(f"  {cat:<16}  {CURRENCY}{daily_avg_cat:.0f} / day")

In [ ]:
# BONUS 2 — Monthly projection
# If I keep spending at this rate, what will my monthly total be?

daily_average = total_spent / len(dates)
monthly_proj  = daily_average * 30
monthly_budget = WEEKLY_BUDGET * 4

print("Monthly projection (based on this week's pattern):")
print(f"  Daily average         : {CURRENCY}{daily_average:.0f}")
print(f"  Projected monthly     : {CURRENCY}{monthly_proj:,.0f}")
print(f"  Monthly budget (est.) : {CURRENCY}{monthly_budget:,.0f}")
diff = monthly_proj - monthly_budget
if diff > 0:
    print(f"  You would be {CURRENCY}{diff:.0f} OVER your monthly budget at this rate.")
else:
    print(f"  You would be {CURRENCY}{abs(diff):.0f} UNDER your monthly budget — good!")

In [ ]:
# BONUS 3 — Most expensive single item in each category

print("Most expensive single purchase per category:")
print()
category_max = defaultdict(lambda: {"amount": 0, "description": "", "date": ""})

for row in expenses:
    cat = row["category"]
    if row["amount"] > category_max[cat]["amount"]:
        category_max[cat] = {
            "amount"     : row["amount"],
            "description": row["description"],
            "date"       : row["date"],
        }

for cat, _ in sorted_cats:
    item = category_max[cat]
    print(f"  {cat:<16} {CURRENCY}{item['amount']:>6.0f}  {item['description']} ({item['date']})")

In [ ]:
# BONUS 4 — What if you cut one category by 20%?
# Simulate the impact of reducing your biggest non-essential category

# Find the biggest non-essential category
essential_cats = {"Food", "Health", "Education"}
non_essential  = [(cat, total) for cat, total in sorted_cats if cat not in essential_cats]

if non_essential:
    target_cat, target_total = non_essential[0]
    saving = target_total * 0.20
    new_total = total_spent - saving

    print(f"Scenario: What if I reduced '{target_cat}' spending by 20%?")
    print()
    print(f"  Current '{target_cat}' spend    : {CURRENCY}{target_total:.0f}")
    print(f"  20% reduction saves       : {CURRENCY}{saving:.0f}")
    print(f"  New weekly total          : {CURRENCY}{new_total:.0f}  (was {CURRENCY}{total_spent:.0f})")
    print(f"  Monthly saving (× 4)      : {CURRENCY}{saving * 4:.0f}")
    print(f"  Yearly saving (× 52)      : {CURRENCY}{saving * 52:,.0f}")

---
## Reflection Questions

Answer these in the cells below — they will help you connect the project to what you learned in Module 1.

1. **Lifecycle check:** Which of the 7 lifecycle steps felt most natural? Which felt most difficult?
2. **Data types:** What is the statistical type of each column in your CSV? (Hint: use the decision tree from Lesson 6.)
3. **The surprise question:** Was there anything in your spending data that genuinely surprised you? If not, does that mean data science added no value here — or does it confirm what you suspected with numbers?
4. **Ethics check:** If a bank could see this data, what could they infer about you beyond just spending habits? (Think about what your categories reveal about your lifestyle.)
5. **If you had more data:** What additional analysis would you want to do with 3 months of data instead of one week?

**Your Answers:**

1. 

2. 

3. 

4. 

5. 

---
## 📌 What You Just Did

You completed a full data science project:

| Step | What you did |
|---|---|
| Define | Wrote 5 specific, answerable questions before touching the data |
| Collect | Loaded a CSV file using Python's `csv` module |
| Clean | Checked for missing values, zero amounts, and category consistency |
| Explore | Computed distributions, per-day counts, transaction size bins |
| Analyse | Answered all 5 questions with specific numbers and visualisations |
| Evaluate | Verified totals match, cross-checked the highest-spend day |
| Communicate | Generated a structured report with written interpretations |

And you did it with **zero external libraries** — pure Python.

In Module 5, you'll do all of this in about 10 lines using Pandas. In Module 7, you'll add beautiful charts. In Module 11, you'll build a model that predicts next week's spending from this week's pattern.

---

## What's Next?

**Module 02 — Python for Data Science**  
You used Python throughout this course — but only the basics. Module 2 builds the full Python foundation: functions, data structures, file handling, and everything else you'll need to work with data at scale.